# E3 — 07A/07B/07C/7D : représentations multihorizons et alignement strict

**Compagnon indépendant des notebooks originaux.** Génère les représentations tabulaires (15 variables), séquentielles (5 × 3) et graphiques (5 × 7 × 3) pour 15, 30, 60 et 120 minutes, avec **exactement les mêmes clés de prédiction pour les trois représentations**. Ne réentraîne aucun modèle et ne modifie aucun fichier initial.

**Important :** le sous-ensemble commun est contraint par la disponibilité simultanée des sept dispositifs ; il peut donc être plus petit que les données tabulaires originales. Les métriques futures devront utiliser uniquement ces clés communes. Les graphes opérationnels sont à reconstruire *sur l'entraînement de chaque horizon* ou à réutiliser uniquement après audit de leur provenance. Aucun graphe n'est fabriqué ici.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve()
SOURCE = ROOT / 'outputs/05_ml_dataset/06_ai_ready_complete_case.csv'
E3 = ROOT / 'outputs/E3_multihorizon'
DEST = ROOT / 'outputs/E3_aligned_representations'
HORIZONS = [15, 30, 60, 120]
VARS = ['temperature','relative_humidity','co2']
SPLITS = ['train','validation','test']
STEP = pd.Timedelta(minutes=15)
assert SOURCE.exists(), SOURCE
raw = pd.read_csv(SOURCE)
raw['timestamp'] = pd.to_datetime(raw['timestamp'],utc=True)
for col in VARS: raw[col] = pd.to_numeric(raw[col],errors='raise')
assert not raw[['uuid','segment_id','timestamp']+VARS].isna().any().any()
assert not raw.duplicated(['uuid','timestamp']).any()
raw = raw.sort_values(['uuid','timestamp']).reset_index(drop=True)
raw_lookup = raw.set_index(['uuid','timestamp'])
DEST.mkdir(parents=True,exist_ok=True)
print('Raw rows:',len(raw),'devices:',raw.uuid.nunique())


In [ ]:
def build_one(h):
    base = E3/f'h{h:03d}'
    f = base/'01_common_supervised_dataset.csv'
    assert f.exists(), f'Execute E3_06_multihorizon_targets first: {f}'
    d = pd.read_csv(f)
    d['timestamp'] = pd.to_datetime(d['timestamp'],utc=True)
    d['target_timestamp'] = pd.to_datetime(d['target_timestamp'],utc=True)
    target = f'target_temperature_t_plus_{h}'
    required = ['uuid','timestamp','target_timestamp','segment_id','split',target]
    assert set(required).issubset(d.columns),set(required)-set(d.columns)
    assert not d.duplicated(['uuid','timestamp']).any()
    assert (d['target_timestamp']-d['timestamp']).eq(pd.Timedelta(minutes=h)).all()
    # Derive historical values from original cleaned series, NOT from target-eligible rows.
    # This avoids losing legitimate history near forecast-boundary exclusions.
    rec=[]
    for row in d.itertuples(index=False):
        u,t,seg,s = row.uuid,row.timestamp,row.segment_id,row.split
        hist_times=[t-STEP*k for k in range(4,-1,-1)]
        keys=pd.MultiIndex.from_tuples([(u,x) for x in hist_times],names=['uuid','timestamp'])
        hist=raw_lookup.reindex(keys)
        if hist[VARS+['segment_id']].isna().any().any(): continue
        if not hist['segment_id'].eq(seg).all(): continue
        # Split purity: earliest lag must be inside the same split; E3 target already within split.
        # Use actual cutoff metadata to avoid inferring split starts from surviving endpoints.
        meta=json.loads((base/'06_split_metadata.json').read_text())
        train_end=pd.Timestamp(meta['train_end']) if 'train_end' in meta else None
        val_end=pd.Timestamp(meta['validation_end']) if 'validation_end' in meta else None
        if train_end is None or val_end is None:
            orig=json.loads((ROOT/'outputs/06_common_experiment/06_split_metadata.json').read_text())
            train_end=pd.Timestamp(orig['train_end']);val_end=pd.Timestamp(orig['validation_end'])
        first=hist_times[0]
        if s=='train' and first>train_end: continue
        if s=='validation' and not (train_end<first<=val_end): continue
        if s=='test' and not (first>val_end): continue
        arr=hist[VARS].to_numpy(dtype=np.float32)
        rec.append((u,t,row.target_timestamp,s,seg,float(getattr(row,target)),arr))
    assert rec, f'No valid historical windows for {h} min'
    devices=sorted(d.uuid.unique().tolist())
    assert len(devices)==7, f'Expected seven devices, got {len(devices)}'
    device_set=set(devices)
    by_time={}
    for r in rec: by_time.setdefault((r[3],r[1]),{})[r[0]]=r
    # All seven devices at each timestamp, plus equal target timestamps.
    complete=[]
    for (s,t),nodes in by_time.items():
        if set(nodes)!=device_set: continue
        targets={nodes[u][2] for u in devices}
        if len(targets)!=1: continue
        complete.append((s,t,nodes))
    complete.sort(key=lambda x:(SPLITS.index(x[0]),x[1]))
    assert complete, f'No complete seven-device snapshots for {h} min'
    out=DEST/f'h{h:03d}';out.mkdir(exist_ok=True)
    pd.DataFrame({'node_index':range(7),'uuid':devices}).to_csv(out/'01_graph_node_index.csv',index=False)
    # Produce one row per node, ordered identically to graph tensor flattening.
    meta_rows=[];tab_rows=[];Xg=[];Yg=[]
    for s,t,nodes in complete:
        x=np.stack([nodes[u][6] for u in devices]) # [7,5,3]
        y=np.array([nodes[u][5] for u in devices],dtype=np.float32)
        Xg.append(x);Yg.append(y)
        for i,u in enumerate(devices):
            r=nodes[u]
            meta_rows.append({'uuid':u,'timestamp':t,'target_timestamp':r[2],'split':s,'node_index':i,'target':y[i]})
            v={'uuid':u,'timestamp':t,'target_timestamp':r[2],'split':s,'target':y[i]}
            for lag in range(5):
                for j,var in enumerate(VARS): v[f'{var}_lag{4-lag}']=float(x[i,lag,j])
            tab_rows.append(v)
    Xg=np.stack(Xg).astype(np.float32);Yg=np.stack(Yg).astype(np.float32)
    metadata=pd.DataFrame(meta_rows);tab=pd.DataFrame(tab_rows)
    assert len(metadata)==len(complete)*7
    assert not metadata.duplicated(['uuid','timestamp']).any()
    # Scale using TRAIN ONLY, shared scaler across tabular/sequence/graph.
    snapshot_splits=np.array([s for s,_,_ in complete])
    trainmask=snapshot_splits=='train'
    assert trainmask.any()
    scaler=StandardScaler().fit(Xg[trainmask].reshape(-1,3))
    Xgs=scaler.transform(Xg.reshape(-1,3)).reshape(Xg.shape).astype(np.float32)
    pd.DataFrame({'variable':VARS,'train_mean':scaler.mean_,'train_scale':scaler.scale_}).to_csv(out/'02_scaler_train_only.csv',index=False)
    tab.to_csv(out/'03_tabular_all_unscaled.csv',index=False)
    metadata.to_csv(out/'04_common_node_keys.csv',index=False)
    audit=[]
    for s in SPLITS:
        ids=np.flatnonzero(snapshot_splits==s)
        assert len(ids)>0, f'Empty split {s} for horizon {h}'
        x=Xg[ids];xs=Xgs[ids];y=Yg[ids]
        # Graph: [snapshots, 5, 7, 3] matching original 07C convention
        x_graph=x.transpose(0,2,1,3); xs_graph=xs.transpose(0,2,1,3)
        np.save(out/f'X_graph_{s}_unscaled.npy',x_graph)
        np.save(out/f'X_graph_{s}_scaled.npy',xs_graph)
        np.save(out/f'y_graph_{s}.npy',y)
        # Sequence and tabular flatten by (timestamp, node_index)
        x_seq=x.reshape(-1,5,3);xs_seq=xs.reshape(-1,5,3);ys=y.reshape(-1)
        np.save(out/f'X_sequence_{s}_unscaled.npy',x_seq)
        np.save(out/f'X_sequence_{s}_scaled.npy',xs_seq)
        np.save(out/f'y_sequence_{s}.npy',ys)
        np.save(out/f'X_tabular_{s}_unscaled.npy',x_seq.reshape(-1,15))
        np.save(out/f'X_tabular_{s}_scaled.npy',xs_seq.reshape(-1,15))
        np.save(out/f'y_tabular_{s}.npy',ys)
        part=metadata[metadata.split==s].reset_index(drop=True)
        part.to_csv(out/f'metadata_{s}.csv',index=False)
        assert len(part)==len(ys)==7*len(ids)
        assert np.array_equal(x_graph.transpose(0,2,1,3).reshape(-1,5,3),x_seq)
        assert np.array_equal(ys,y.reshape(-1))
        audit.append({'horizon_min':h,'split':s,'graph_snapshots':len(ids),'node_samples':len(part)})
    (out/'05_metadata.json').write_text(json.dumps({'horizon_minutes':h,'devices':devices,'history_steps':5,'features':VARS,'target':target,'graph_shape':'[snapshots, 5, 7, 3]','common_keys':'uuid,timestamp,target_timestamp,split','training_only_scaler':True,'note':'No graph edges constructed; E3 training must build training-only topologies.'},indent=2),encoding='utf-8')
    return audit, {'horizon':h,'eligible_targets':len(d),'history_valid':len(rec),'aligned_node_samples':len(metadata),'snapshots':len(complete)}

all_audits=[];summaries=[]
for h in HORIZONS:
    audit,summary=build_one(h)
    all_audits+=audit;summaries.append(summary)
    print('Horizon',h,'min:',summary)
pd.DataFrame(all_audits).to_csv(DEST/'00_split_alignment_audit.csv',index=False)
pd.DataFrame(summaries).to_csv(DEST/'00_horizon_summary.csv',index=False)
print('Saved to:',DEST)


## Contrôles et limites avant entraînement

1. Vérifier les effectifs à 15 minutes avec le protocole historique `7D` (64 701 / 13 825 / 13 790). **Ne pas supposer leur égalité** : l'algorithme ci-dessus impose un sous-ensemble de timestamps où les sept dispositifs sont présents et exige la continuité des historiques.
2. Ce notebook utilise uniquement les 15 valeurs historiques des trois variables, **sans variables calendaires**. Cela correspond à la comparaison *core 15*, pas au jeu tabulaire enrichi de variables calendaires.
3. Le GCN peut recevoir les observations des sept dispositifs si des arêtes inter-dispositifs sont activées. Les entrées par nœud sont identiques, mais l'information accessible après propagation ne l'est pas : comparer également le graphe natif sans arêtes.
4. Les nouvelles expériences doivent reconstruire les graphes opérationnels avec **l'entraînement de chaque horizon seulement** et utiliser la même liste de dispositifs et les mêmes clés.
5. Les métriques multi-horizons devront inclure la persistance, les différences de RMSE relativement à la persistance et les comparaisons appariées sur les mêmes échantillons.
6. Aucune métrique de prévision n'est produite par ce notebook.